In [ ]:
import importlib.util
import re
import sys
import threading
from pathlib import Path

import torch
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    PreTrainedTokenizerFast,
    TextIteratorStreamer,
)


In [ ]:
PROJECT_ROOT = Path("..")
MODEL_DIR = PROJECT_ROOT / "model" / "GenNA"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" else torch.float32

flash_attn_available = (
    device.type == "cuda" and importlib.util.find_spec("flash_attn") is not None
)

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Required path not found: {MODEL_DIR}")

print(f"Device: {device}")
print(f"Dtype: {dtype}")
print(f"flash_attn available: {flash_attn_available}")
print(f"Model directory: {MODEL_DIR}")


In [ ]:
tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_DIR)

print({
    "vocab_size": tokenizer.vocab_size,
    "pad_token": tokenizer.pad_token,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token": tokenizer.eos_token,
    "eos_token_id": tokenizer.eos_token_id,
})


In [ ]:
config = AutoConfig.from_pretrained(MODEL_DIR)

model_kwargs = {
    "config": config,
    "torch_dtype": dtype,
}

if flash_attn_available:
    model_kwargs["attn_implementation"] = "flash_attention_2"

model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, **model_kwargs).to(device)
model.eval()

print(type(model).__name__)


In [ ]:
prompt = "RNA, Nannospalax galili, Emc9, ER membrane protein complex subunit 9<seq>"

inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
inputs = {key: value.to(device) for key, value in inputs.items()}

print(prompt)
inputs


In [ ]:
streamer = TextIteratorStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

gen_kwargs = {
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs.get("attention_mask"),
    "max_new_tokens": 1000,
    "do_sample": True,
    "top_p": 0.8,
    "temperature": 0.7,
    "top_k": 0,
    "repetition_penalty": 1.3,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "streamer": streamer,
}

def run_generation():
    with torch.inference_mode():
        model.generate(**gen_kwargs)


sys.stdout.write(prompt)
sys.stdout.flush()

thread = threading.Thread(target=run_generation, daemon=True)
thread.start()

for chunk in streamer:
    chunk = chunk.replace("Ġ", "")
    chunk = re.sub(r"\s+", "", chunk)
    sys.stdout.write(chunk)
    sys.stdout.flush()

thread.join()
print()
